<a href="https://colab.research.google.com/github/Uday-Naik-coder/Explainable-Misinformation-Detection-using-DANN-and-LIME-models/blob/test/misinformaion_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# ROBUST FAKE NEWS DETECTOR (Auto-Cleaning + Linear DANN)
# =============================================================================

import os, re, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from lime.lime_text import LimeTextExplainer

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================
class Config:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    SEED = 42
    CSV_PATH = "misovac_dataset.csv"  # Uses your original file

    # Hyperparameters optimized for small datasets
    HIDDEN_DIM = 64
    DROPOUT = 0.3
    LEARNING_RATE = 1e-3
    WEIGHT_DECAY = 1e-3    # Strong regularization
    BATCH_SIZE = 16        # Small batch size for stability
    EPOCHS = 30
    NOISE_LEVEL = 0.05     # Data Augmentation strength

config = Config()
print(f"Running on: {config.DEVICE}")

# Set seeds
random.seed(config.SEED)
np.random.seed(config.SEED)
torch.manual_seed(config.SEED)

# =============================================================================
# 1. ROBUST DATA LOADING & CLEANING
# =============================================================================
def clean_text(text):
    """Aggressively removes artifacts to force semantic learning."""
    text = str(text)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    # Remove Handles and Hashtags (content remains, tags go)
    text = re.sub(r'\@\w+', '', text)
    text = re.sub(r'\#\w+', '', text)
    # Remove Dataset Artifacts (The root cause of overfitting)
    artifacts = ["RT", "Edit:", "Link in description", "Subscribe", "TL;DR", "r/COVID19"]
    for art in artifacts:
        text = text.replace(art, "")
    # Remove non-ascii
    text = text.encode('ascii', 'ignore').decode('ascii')
    return text.strip()

def load_data(path):
    print("📂 Loading and Cleaning Data...")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Could not find {path}")

    df = pd.read_csv(path)
    df.columns = [c.lower().strip() for c in df.columns]

    # Fix Labels (1=Real, 0=Fake based on your dataset analysis)
    label_map = {
        1: "real", 0: "fake",
        "1": "real", "0": "fake",
        "real": "real", "fake": "fake"
    }
    df['label'] = df['label'].map(label_map)

    # Apply Cleaning
    df['text'] = df['text'].apply(clean_text)

    # Remove empty rows after cleaning
    df = df[df['text'].str.len() > 10]

    # Strict Deduplication (Crucial!)
    before = len(df)
    df = df.drop_duplicates(subset=['text', 'label'])
    print(f"✓ Cleaned & Deduplicated: {before} -> {len(df)} unique semantic samples")

    return df

# =============================================================================
# 2. MODEL COMPONENTS
# =============================================================================
class EmbeddingManager:
    def __init__(self):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")

    def get_embeddings(self, texts):
        return self.model.encode(texts, convert_to_numpy=True, show_progress_bar=False)

class EmbDataset(Dataset):
    def __init__(self, X, y, d, noise_level=0.0):
        self.X = X
        self.y = y
        self.d = d
        self.noise_level = noise_level

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        emb = self.X[i]
        # Data Augmentation: Add noise to embeddings
        if self.noise_level > 0:
            noise = np.random.normal(0, self.noise_level, emb.shape)
            emb = emb + noise
        return torch.tensor(emb).float(), torch.tensor(self.y[i]).long(), torch.tensor(self.d[i]).long()

class LinearDANN(nn.Module):
    """Simplified DANN for better generalization on small data."""
    def __init__(self, input_dim, hidden_dim, n_labels, n_domains):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT)
        )
        self.label_classifier = nn.Linear(hidden_dim, n_labels)
        self.domain_classifier = nn.Linear(hidden_dim, n_domains)

    def forward(self, x, grl_coeff=1.0):
        features = self.feature_extractor(x)
        # Gradient Reversal
        reversed_features = features * -grl_coeff + features.detach() * (1 + grl_coeff)
        return self.label_classifier(features), self.domain_classifier(reversed_features)

# =============================================================================
# 3. TRAINING LOOP
# =============================================================================
def train_and_test():
    # 1. Data Prep
    df = load_data(config.CSV_PATH)

    emb_manager = EmbeddingManager()
    print("🔢 Generating embeddings...")
    embeddings = emb_manager.get_embeddings(df['text'].tolist())

    le_label = LabelEncoder()
    le_domain = LabelEncoder()
    y = le_label.fit_transform(df['label'])
    d = le_domain.fit_transform(df['platform'].fillna('unknown'))

    # Split
    X_train, X_test, y_train, y_test, d_train, d_test = train_test_split(
        embeddings, y, d, test_size=0.2, random_state=config.SEED, stratify=y
    )

    # Datasets (Noise only on training)
    train_ds = EmbDataset(X_train, y_train, d_train, noise_level=config.NOISE_LEVEL)
    test_ds = EmbDataset(X_test, y_test, d_test, noise_level=0.0)

    train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=config.BATCH_SIZE)

    # Model
    model = LinearDANN(384, config.HIDDEN_DIM, len(le_label.classes_), len(le_domain.classes_)).to(config.DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()

    print("🏋️  Training...")
    for epoch in range(config.EPOCHS):
        model.train()
        for x, label, domain in train_loader:
            x, label, domain = x.to(config.DEVICE), label.to(config.DEVICE), domain.to(config.DEVICE)

            p = epoch / config.EPOCHS
            grl_coeff = 2.0 / (1.0 + np.exp(-10 * p)) - 1.0

            optimizer.zero_grad()
            l_out, d_out = model(x, grl_coeff)
            loss = criterion(l_out, label) + 0.1 * criterion(d_out, domain)
            loss.backward()
            optimizer.step()

    # Evaluation
    model.eval()
    preds = []
    with torch.no_grad():
        for x, _, _ in test_loader:
            out, _ = model(x.to(config.DEVICE))
            preds.extend(torch.argmax(out, 1).cpu().numpy())

    print("\n📊 Test Set Report:")
    print(classification_report(y_test, preds, target_names=le_label.classes_))

    return model, emb_manager, le_label

# =============================================================================
# 4. INTERACTIVE TESTER
# =============================================================================
if __name__ == "__main__":
    model, emb_manager, le_label = train_and_test()
    explainer = LimeTextExplainer(class_names=le_label.classes_)

    print("\n" + "="*60)
    print("🧪  ROBUST FAKE NEWS TESTER")
    print("   (The model now ignores 'Edit:', 'RT', etc.)")
    print("="*60)

    while True:
        text = input("\n📝 Enter news (or 'q' to quit): ")
        if text.lower() in ['q', 'exit']: break

        # Clean the input exactly like the training data
        clean_input = clean_text(text)

        # Predict
        emb = emb_manager.get_embeddings([clean_input])[0]
        model.eval()
        with torch.no_grad():
            x = torch.tensor(emb).float().unsqueeze(0).to(config.DEVICE)
            out, _ = model(x, 0.0)
            probs = torch.softmax(out, 1).cpu().numpy()[0]

        pred_idx = np.argmax(probs)
        pred_label = le_label.classes_[pred_idx]
        conf = probs[pred_idx]

        print(f"Prediction: {pred_label.upper()} ({conf:.2%})")

        # Explain
        try:
            exp = explainer.explain_instance(
                clean_input,
                lambda t: torch.softmax(model(torch.tensor(emb_manager.get_embeddings(t)).float().to(config.DEVICE), 0.0)[0], 1).detach().cpu().numpy(),
                num_features=6
            )
            print("Why? (Top words):")
            for feature, weight in exp.as_list():
                direction = "REAL" if weight > 0 else "FAKE" # Adjust based on your label index 0/1
                if le_label.classes_[0] == "real": direction = "FAKE" if weight > 0 else "REAL"

                print(f"  {feature:<15} {weight:+.4f}")
        except Exception as e:
            pass


Running on: cuda
📂 Loading and Cleaning Data...
✓ Cleaned & Deduplicated: 1000 -> 39 unique semantic samples
🔢 Generating embeddings...
🏋️  Training...

📊 Test Set Report:
              precision    recall  f1-score   support

        fake       1.00      1.00      1.00         4
        real       1.00      1.00      1.00         4

    accuracy                           1.00         8
   macro avg       1.00      1.00      1.00         8
weighted avg       1.00      1.00      1.00         8


🧪  ROBUST FAKE NEWS TESTER
   (The model now ignores 'Edit:', 'RT', etc.)
Prediction: FAKE (61.72%)
Why? (Top words):
  apple           -0.0415
  cause           -0.0384
  covid19         +0.0278
